# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [12]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.71 GB
MemFree: 140.44 GB
MemAvailable: 512.83 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved

# 2. Load CoQA Dataset

## 2.1 Download CoQA

In [13]:
import os
import requests
import json
from pathlib import Path
from tqdm import tqdm
import hashlib

def download_coqa_dataset(
    save_dir: str = "/nfs/students/daro/data/CoQA",
    version: str = "dev",
    force_download: bool = False
) -> str:
    """
    Downloads the CoQA dataset and returns the path to the downloaded file.
    
    Args:
        save_dir: Directory to save the dataset
        version: Dataset version ('dev' or 'train')
        force_download: If True, redownload even if file exists
    
    Returns:
        Path to the downloaded JSON file
    """
    # Create save directory if it doesn't exist
    save_dir = Path(save_dir)
    
    # Define URLs and expected MD5 hashes for verification
    COQA_URLS = {
        'dev': 'https://downloads.cs.stanford.edu/nlp/data/coqa/coqa-dev-v1.0.json',
        'train': 'https://downloads.cs.stanford.edu/nlp/data/coqa/coqa-train-v1.0.json'
    }
    
    COQA_MD5 = {
        'dev': 'c67e51d92439bfc3c4401b670305e46f',
        'train': 'b0fdb2bc1bd4dd79b2590d2aff70e9a2'
    }
    
    if version not in COQA_URLS:
        raise ValueError(f"Invalid version: {version}. Must be one of {list(COQA_URLS.keys())}")
    
    url = COQA_URLS[version]
    filename = url.split('/')[-1]
    save_path = save_dir / filename
    
    # Check if file already exists and verify its integrity
    if save_path.exists() and not force_download:
        print(f"Found existing file at {save_path}")
        # Verify file integrity
        with open(save_path, 'rb') as f:
            file_hash = hashlib.md5(f.read()).hexdigest()
        if file_hash == COQA_MD5[version]:
            print("File integrity verified.")
            return str(save_path)
        else:
            print("File integrity check failed. Re-downloading...")
    
    # Download the file
    print(f"Downloading CoQA {version} dataset from {url}")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    # Get file size for progress bar
    file_size = int(response.headers.get('content-length', 0))
    
    # Download with progress bar
    with open(save_path, 'wb') as f, tqdm(
        desc=filename,
        total=file_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as pbar:
        for data in response.iter_content(chunk_size=1024):
            size = f.write(data)
            pbar.update(size)
    
    # # Verify downloaded file
    # with open(save_path, 'rb') as f:
    #     file_hash = hashlib.md5(f.read()).hexdigest()
    # if file_hash != COQA_MD5[version]:
    #     raise ValueError("Downloaded file failed integrity check")
    
    print(f"Successfully downloaded and verified CoQA {version} dataset")
    
    # Quick validation of JSON format
    try:
        with open(save_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            print(f"Dataset contains {len(data['data'])} stories")
    except (json.JSONDecodeError, KeyError) as e:
        raise ValueError(f"Downloaded file is not in valid CoQA format: {e}")
    
    return str(save_path)

# Download dev set
dev_path = download_coqa_dataset(version='dev')
print(f"Dataset downloaded to: {dev_path}")

# Download train set
train_path = download_coqa_dataset(version='train')
print(f"Dataset downloaded to: {train_path}")

# Force redownload if needed
dev_path = download_coqa_dataset(version='dev', force_download=True)

coqa-dev-v1.0.json: 100%|██████████| 8.67M/8.67M [00:01<00:00, 6.33MiB/s]


Successfully downloaded and verified CoQA dev dataset
Dataset contains 500 stories
Dataset downloaded to: /nfs/students/daro/data/CoQA/coqa-dev-v1.0.json


coqa-train-v1.0.json: 100%|██████████| 46.7M/46.7M [00:06<00:00, 7.39MiB/s]


Successfully downloaded and verified CoQA train dataset
Dataset contains 7199 stories
Dataset downloaded to: /nfs/students/daro/data/CoQA/coqa-train-v1.0.json


coqa-dev-v1.0.json: 100%|██████████| 8.67M/8.67M [00:01<00:00, 7.13MiB/s]


Successfully downloaded and verified CoQA dev dataset
Dataset contains 500 stories


## 2.2 Load setup

In [16]:
import json
import random
from typing import List, Tuple, Dict, Optional
from pathlib import Path

# Import the typo modification function from the existing codebase
from src.reliability.apply_typos import apply_typo_modifications

def create_typo_dict(typo_type: str, intensity: int) -> Dict[str, int]:
    """
    Creates a dictionary of typo modifications with specified intensity.
    Reuses the same typo types as the original implementation.
    """
    base_dict = {
        "char_insertion": 0,
        "char_deletion": 0,
        "char_replacement": 0,
        "char_repetition": 0,
        "char_swapping": 0,
        "word_CMW": 0,
        "char_LCC": 0,
        "word_synonym": 0,
        "char_insert_noise": 0,
        "word_repeat": 0,
        "char_substitution": 0,
        "word_emoji": 0,
        "word_internet_slang": 0,
        "word_phrase_translation": 0,
        "word_context_aware_insertion": 0,
        "word_remove_punctuation": 0,
        "word_keyword_only": 0,
        "word_taxonomy_pos": 0,
        "word_taxonomy_neg": 0
    }
    
    if typo_type in base_dict:
        base_dict[typo_type] = intensity
    elif typo_type == "random":
        for _ in range(intensity):
            key = random.choice(list(base_dict.keys()))
            base_dict[key] += 1
            
    return base_dict

def load_coqa_dataset(
    file_path: str,
    max_relations: int = 1,
    max_entries: Optional[int] = None,
    typo_type: str = "none",
    typo_intensity: int = 0
) -> List[Tuple[str, str]]:
    """
    Loads the CoQA dataset and returns a list of (question, answer) tuples.
    Applies typo modifications if specified.
    
    Args:
        file_path: Path to the CoQA JSON file
        max_relations: Maximum number of relations to consider (kept for API compatibility)
        max_entries: Maximum number of QA pairs to load (None for all)
        typo_type: Type of typo to apply ("none" for no typos)
        typo_intensity: Intensity of typo modifications
    
    Returns:
        List of (question, answer) tuples
    """
    qa_pairs = []
    
    # Load the CoQA dataset
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Process each story in the dataset
    for story_idx, story in enumerate(data['data']):
        if max_entries and len(qa_pairs) >= max_entries:
            break
            
        questions = story['questions']
        answers = story['answers']
        
        # Create QA pairs for each story
        for q, a in zip(questions, answers):
            question = q['input_text']
            answer = a['input_text']
            
            # Apply typo modifications if specified
            if typo_type != "none":
                typo_dict = create_typo_dict(typo_type, typo_intensity)
                # Pass the answer as part of the context for typo modifications
                question = apply_typo_modifications(question, typo_dict, [answer])
                
            qa_pairs.append((question, answer))
            
            if max_entries and len(qa_pairs) >= max_entries:
                break
                
    return qa_pairs

# Debugging script for Jupyter notebook
def debug_coqa_loading():
    """
    Debug function to test CoQA dataset loading with various parameters
    """
    # Example path - modify according to your setup
    coqa_path = "/nfs/students/daro/data/CoQA/coqa-dev-v1.0.json"
    
    # Test different loading configurations
    test_configs = [
        {"max_entries": 5, "typo_type": "none", "typo_intensity": 0},
        {"max_entries": 3, "typo_type": "char_insertion", "typo_intensity": 1},
        {"max_entries": 3, "typo_type": "random", "typo_intensity": 2}
    ]
    
    for config in test_configs:
        print(f"\nTesting configuration: {config}")
        qa_pairs = load_coqa_dataset(
            coqa_path,
            max_entries=config['max_entries'],
            typo_type=config['typo_type'],
            typo_intensity=config['typo_intensity']
        )
        
        # Print the first few QA pairs
        for idx, (question, answer) in enumerate(qa_pairs):
            print(f"\nPair {idx + 1}:")
            print(f"Q: {question}")
            print(f"A: {answer}")

## 2.2 Loading script

In [17]:
# Run debug function
debug_coqa_loading()

# Or load with specific parameters
qa_pairs = load_coqa_dataset(
    "/nfs/students/daro/data/CoQA/coqa-dev-v1.0.json",
    max_entries=10,
    typo_type="char_insertion",
    typo_intensity=1
)

# Print first few pairs
for idx, (q, a) in enumerate(qa_pairs[:3]):
    print(f"\nPair {idx + 1}:")
    print(f"Q: {q}")
    print(f"A: {a}")


Testing configuration: {'max_entries': 5, 'typo_type': 'none', 'typo_intensity': 0}

Pair 1:
Q: What color was Cotton?
A: white

Pair 2:
Q: Where did she live?
A: in a barn

Pair 3:
Q: Did she live alone?
A: no

Pair 4:
Q: Who did she live with?
A: with her mommy and 5 sisters

Pair 5:
Q: What color were her sisters?
A: orange and white

Testing configuration: {'max_entries': 3, 'typo_type': 'char_insertion', 'typo_intensity': 1}

Pair 1:
Q: What coflor was Cotton?
A: white

Pair 2:
Q: Whqere did she live?
A: in a barn

Pair 3:
Q: Did she Slive alone?
A: no

Testing configuration: {'max_entries': 3, 'typo_type': 'random', 'typo_intensity': 2}

Pair 1:
Q: What color it was CottoN?
A: white

Pair 2:
Q: Wheere did she live?
A: in a barn

Pair 3:
Q: Did she £ive alone?
A: no

Pair 1:
Q: What coloYr was Cotton?
A: white

Pair 2:
Q: Where did shhe live?
A: in a barn

Pair 3:
Q: Did she live aelone?
A: no
